# Ancient Tamil (Palm-Leaf Manuscript) Character Recognition
## Implementation of Research Papers: FFBNN (2014) & MABPN (2020)
### தமிழ் பனையோலை கையெழுத்துப் பிரதி எழுத்துணர்தல் (*Panaiyolai Suvadigal OCR*)

---

> **Research Foundations:**
> 1. **Paper 1:** Ramya & Parvathavarthini (2014) — *Feed Forward Back Propagation Neural Network (FFBNN)* with Binary Coded Features.
> 2. **Paper 2:** Poornima Devi & Sornam (2020) — *Modified Adaptive Backpropagation Network (MABPN)* with GLCM Features, Nguyen-Widrow Initialization, and Shannon Activation.
> 3. **Comparative Baseline:** Sundaram et al. & Narang et al. (2020) (SIFT + Gabor + PCA Pipeline).
>
> **Strict Constraint:** Classical Machine Learning & Shallow Neural Networks only. **Zero Deep Learning (No CNNs, No PyTorch, No TensorFlow, No GPUs).**

---

## 🚀 HOW TO RUN THIS NOTEBOOK

### In Visual Studio Code:
1. Open this file: `ancient_tamil_recognition.ipynb` in VS Code.
2. Select your Python kernel in the top-right corner (Python 3.10+ with standard requirements installed).
3. Click **"Run All"** at the top of the notebook window to execute all experiments sequentially, OR press **Shift + Enter** on each cell one by one.

### In Jupyter Notebook / Lab (Browser):
```powershell
cd c:\Users\HARSHIT\Documents\ML\ancient_tamil_ocr
jupyter notebook ancient_tamil_recognition.ipynb
```
Click **Kernel -> Restart & Run All**.

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure UTF-8 output encoding for Tamil glyphs on Windows
if sys.platform.startswith('win'):
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

# Configure inline plotting and Tamil fonts
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = ['Nirmala UI', 'Latha', 'Vijaya', 'DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# Import project modules
import config
import dataset
import preprocess
import features
import neural_networks
import models
import evaluate
import predict
import process_manuscript

print('All modules loaded successfully!')

## 1. Dataset Loading: 10 Ancient Tamil Character Classes
Loads 500 samples across 10 classes with an 80/20 stratified train/test split (400 train, 100 test).
Glyphs: `அ, ஆ, இ, க, ச, த, ப, ம, ர, ல`.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test, class_names, file_paths = dataset.load_character_dataset(
    data_dir=config.DATA_DIR,
    test_size=config.TRAIN_TEST_SPLIT_RATIO,
    random_state=config.RANDOM_SEED,
    auto_generate_if_missing=True
)

print(f"Total Character Classes:     {len(class_names)}")
print(f"Training Partition (80%):    {len(X_train_raw)} samples (40 per class)")
print(f"Held-Out Test Set (20%):     {len(X_test_raw)} samples (10 per class)")

# Display representative character from each class
fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for i, cls in enumerate(class_names):
    ax = axes[i // 5, i % 5]
    sample_idx = np.where(y_train == i)[0][0]
    glyph = config.TAMIL_GLYPH_MAP.get(cls, '')
    ax.imshow(X_train_raw[sample_idx])
    ax.set_title(f"{cls} ({glyph})", fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle('Ancient Tamil Palm-Leaf Character Classes (Panaiyolai Suvadigal)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Manuscript Preprocessing & Sauvola Binarization
Restores degraded palm leaves through a 6-stage pipeline:
1. **Grayscale conversion**
2. **Resizing to 64$\times$64**
3. **Gaussian smoothing** (fiber grain reduction)
4. **CLAHE** (Contrast Limited Adaptive Histogram Equalization)
5. **Sauvola adaptive binarization** ($T = m[1 + k(s/R - 1)]$, isolates soot ink from leaf background)
6. **Morphological opening & closing** (removes speckles & bridges stroke cracks)

In [ ]:
# Preprocess the entire dataset
X_train_prep = preprocess.preprocess_dataset(X_train_raw)
X_test_prep = preprocess.preprocess_dataset(X_test_raw)

# Display intermediate restoration stages for 5 sample classes
sample_indices = [np.where(y_train == c)[0][0] for c in range(5)]
fig, axes = plt.subplots(5, 5, figsize=(14, 12))
stage_keys = ['Original', 'Grayscale', 'CLAHE Enhanced', 'Sauvola Binarized', 'Morph Cleaned']
stage_titles = ['1. Raw Palm Leaf', '2. Grayscale (64x64)', '3. CLAHE', '4. Sauvola Binary', '5. Morph Cleaned']

for i, s_idx in enumerate(sample_indices):
    stages = preprocess.get_pipeline_stages(X_train_raw[s_idx])
    cls_name = class_names[y_train[s_idx]]
    glyph = config.TAMIL_GLYPH_MAP.get(cls_name, '')
    for j, key in enumerate(stage_keys):
        ax = axes[i, j]
        if j == 0:
            ax.imshow(stages[key])
            ax.set_ylabel(f"{cls_name} ({glyph})", fontsize=11, fontweight='bold', color='#2c3e50')
        else:
            ax.imshow(stages[key], cmap='gray')
        if i == 0:
            ax.set_title(stage_titles[j], fontsize=11, fontweight='semibold')
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('Step-by-Step Palm-Leaf Manuscript Character Restoration', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Paper 1: Ramya & Parvathavarthini (2014) — FFBNN Pipeline
**Methodology:**
- **Feature Representation:** Resamples character crops to a uniform 8$\times$8 grid producing a **64-dimensional binary coded vector** $\mathbf{x} \in \{0, 1\}^{64}$.
- **Classifier:** Feed Forward Back Propagation Neural Network (**FFBNN**).
- **Architecture:** Input ($64$) $\rightarrow$ Hidden Layer ($64$ neurons with Sigmoid $\sigma(z)$) $\rightarrow$ Output Layer ($10$ classes with Softmax).
- **Optimization:** Standard gradient descent backpropagation with momentum ($\alpha=0.85$).

In [ ]:
# 1. Extract Binary Coded Features (Paper 1)
pipe_bin = features.FeaturePipeline(feature_type="binary_coded", random_state=config.RANDOM_SEED)
X_tr_bin = pipe_bin.fit_transform(X_train_prep)
X_te_bin = pipe_bin.transform(X_test_prep)
print(f"Paper 1 Feature Dimension: {X_tr_bin.shape[1]} attributes per sample")

# 2. Train Feed Forward Back Propagation Neural Network (FFBNN)
ffbnn = neural_networks.FeedForwardBackpropNN(
    hidden_layer_sizes=(64,),
    learning_rate=0.05,
    momentum=0.85,
    max_iter=350,
    random_state=config.RANDOM_SEED
)
ffbnn.fit(X_tr_bin, y_train)

# 3. Evaluate on Held-Out Test Set
preds_ffbnn = ffbnn.predict(X_te_bin)
metrics_p1 = evaluate.evaluate_predictions(y_test, preds_ffbnn, class_names)

print("="*55)
print(" PAPER 1 RESULTS: FFBNN + Binary Coded Features")
print("="*55)
print(f" Test Accuracy: {metrics_p1['accuracy']*100:.2f}%")
print(f" Macro F1:      {metrics_p1['f1_macro']*100:.2f}%")
print(f" Macro Precision:{metrics_p1['precision_macro']*100:.2f}%")
print(f" Macro Recall:   {metrics_p1['recall_macro']*100:.2f}%")
print("="*55)

# Plot Training Loss Progression Curve
plt.figure(figsize=(8, 4))
plt.plot(ffbnn.loss_history_, color='#e74c3c', lw=2.2, label='Cross-Entropy Loss')
plt.title('Paper 1: FFBNN Training Loss Curve (Ramya & Parvathavarthini, 2014)', fontsize=12, fontweight='bold')
plt.xlabel('Epoch', fontsize=10, fontweight='semibold')
plt.ylabel('Loss', fontsize=10, fontweight='semibold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Paper 2: Poornima Devi & Sornam (2020) — MABPN + GLCM Pipeline
**Methodology:**
- **Feature Representation:** Gray-Level Co-occurrence Matrix (**GLCM**) texture features capturing second-order spatial statistics: **Contrast, Correlation, Energy, Homogeneity** across $0^\circ, 45^\circ, 90^\circ, 135^\circ$ and spatial quadrants ($64$-D).
- **Classifier:** Modified Adaptive Backpropagation Network (**MABPN**).
- **Key Optimizations over Paper 1:**
  1. **Nguyen-Widrow Initialization:** Scales weights by $\beta = 0.7 \cdot H^{1/D}$ to distribute active regions evenly over input space and prevent saturation.
  2. **Shannon Activation Function:** $f(x) = \sin(x) - \sin(\sin(x))$ acts as a localized wavelet-like pattern detector.
  3. **Adaptive Learning Rate:** $\eta(t) = \eta(t-1) \times 1.05$ when error decreases, $\eta(t) = \eta(t-1) \times 0.70$ when error spikes.

In [ ]:
# 1. Extract GLCM Texture Features (Paper 2)
pipe_glcm = features.FeaturePipeline(feature_type="glcm", random_state=config.RANDOM_SEED)
X_tr_glcm = pipe_glcm.fit_transform(X_train_prep)
X_te_glcm = pipe_glcm.transform(X_test_prep)
print(f"Paper 2 Feature Dimension: {X_tr_glcm.shape[1]} attributes per sample")

# 2. Train Modified Adaptive Backpropagation Network (MABPN)
mabpn = neural_networks.ModifiedAdaptiveBackpropNN(
    hidden_layer_sizes=(64,),
    initial_learning_rate=0.05,
    lr_inc=1.05,
    lr_dec=0.70,
    activation="shannon",
    max_iter=350,
    random_state=config.RANDOM_SEED
)
mabpn.fit(X_tr_glcm, y_train)

# 3. Evaluate on Held-Out Test Set
preds_mabpn = mabpn.predict(X_te_glcm)
metrics_p2 = evaluate.evaluate_predictions(y_test, preds_mabpn, class_names)

print("="*55)
print(" PAPER 2 RESULTS: MABPN + GLCM Texture Features")
print("="*55)
print(f" Test Accuracy: {metrics_p2['accuracy']*100:.2f}%")
print(f" Macro F1:      {metrics_p2['f1_macro']*100:.2f}%")
print(f" Macro Precision:{metrics_p2['precision_macro']*100:.2f}%")
print(f" Macro Recall:   {metrics_p2['recall_macro']*100:.2f}%")
print("="*55)

# Plot Loss Progression and Adaptive Learning Rate History
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(mabpn.loss_history_, color='#2980b9', lw=2.2, label='Cross-Entropy Loss')
ax1.set_title('MABPN Loss Curve with Shannon Activation', fontsize=11, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=10, fontweight='semibold')
ax1.set_ylabel('Loss', fontsize=10, fontweight='semibold')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend()

ax2.plot(mabpn.lr_history_, color='#27ae60', lw=2.2, label='Adaptive Learning Rate (η)')
ax2.set_title('Dynamic Learning Rate Adaptation', fontsize=11, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=10, fontweight='semibold')
ax2.set_ylabel('Learning Rate η', fontsize=10, fontweight='semibold')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend()

plt.suptitle('Paper 2: MABPN Training Dynamics (Poornima Devi & Sornam, 2020)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Comparative Benchmarking: Paper 1 vs Paper 2 vs Baseline Models
Direct empirical comparison of:
- Paper 1: FFBNN (Binary Coded)
- Paper 2: MABPN (GLCM Texture)
- Classical Shallow MLP (Gabor + SIFT + PCA)
- Polynomial Kernel SVM (Narang et al. baseline)
- All Combined Features + SVM-Linear

In [ ]:
# Train baseline Polynomial SVM and Shallow MLP on Gabor+SIFT
pipe_gs = features.FeaturePipeline(feature_type="gabor_sift", random_state=config.RANDOM_SEED)
X_tr_gs = pipe_gs.fit_transform(X_train_prep)
X_te_gs = pipe_gs.transform(X_test_prep)

from sklearn.decomposition import PCA
pca = PCA(n_components=config.PCA_VARIANCE_RATIO, random_state=config.RANDOM_SEED)
X_tr_pca = pca.fit_transform(X_tr_gs)
X_te_pca = pca.transform(X_te_gs)

# 1. SVM-Poly
clf_poly = models.get_classifiers(random_state=config.RANDOM_SEED)["SVM-Poly"]
clf_poly.fit(X_tr_pca, y_train)
acc_svm = np.mean(clf_poly.predict(X_te_pca) == y_test) * 100.0

# 2. Classical Shallow MLP (1 Layer)
clf_mlp = models.get_classifiers(random_state=config.RANDOM_SEED)["MLP (Shallow NN)"]
clf_mlp.fit(X_tr_pca, y_train)
acc_mlp = np.mean(clf_mlp.predict(X_te_pca) == y_test) * 100.0

# Build Comparison DataFrame
df_comp = pd.DataFrame([
    {"Framework": "Paper 1: Ramya & Parvathavarthini (2014)", "Model": "FFBNN (Custom)", "Feature Set": "Binary Coded (8x8)", "Accuracy (%)": metrics_p1['accuracy']*100.0},
    {"Framework": "Paper 2: Poornima Devi & Sornam (2020)", "Model": "MABPN (Nguyen-Widrow+Shannon)", "Feature Set": "GLCM Texture", "Accuracy (%)": metrics_p2['accuracy']*100.0},
    {"Framework": "Narang et al. / Sundaram Baseline", "Model": "Polynomial SVM (d=3)", "Feature Set": "Gabor + SIFT + PCA", "Accuracy (%)": acc_svm},
    {"Framework": "Classical Shallow ANN", "Model": "Shallow MLP (L-BFGS)", "Feature Set": "Gabor + SIFT + PCA", "Accuracy (%)": acc_mlp},
])

display(df_comp)

# Plot Comparative Bar Chart
plt.figure(figsize=(10, 5))
bars = plt.bar(df_comp["Framework"], df_comp["Accuracy (%)"], color=['#e74c3c', '#2980b9', '#f39c12', '#27ae60'], edgecolor='#2c3e50', width=0.55)
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, h + 1.2, f"{h:.1f}%", ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.ylim(0, 105)
plt.title('Empirical Comparison: Research Papers vs Baseline Frameworks', fontsize=13, fontweight='bold', pad=12)
plt.ylabel('Held-Out Test Accuracy (%)', fontsize=11, fontweight='semibold')
plt.xticks(rotation=15, ha='right', fontsize=9, fontweight='semibold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 6. Confusion Matrix & Paleographic Error Analysis
Visualizing class-by-class recall and identifying why specific ancient Tamil characters are confused by classifiers due to physical palm-leaf fiber decay.

In [ ]:
# Confusion Matrix for the winning baseline model (SVM-Poly + Gabor/SIFT)
test_preds = clf_poly.predict(X_te_pca)
cm = evaluate.evaluate_predictions(y_test, test_preds, class_names)['confusion_matrix']
cm_norm = cm.astype(np.float32) / np.maximum(1, np.sum(cm, axis=1, keepdims=True))
display_labels = [f"{c}\n({config.TAMIL_GLYPH_MAP.get(c, '')})" for c in class_names]

plt.figure(figsize=(9, 7))
annot = np.empty_like(cm, dtype=object)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        annot[i, j] = f"{cm[i, j]}\n({cm_norm[i, j]*100:.0f}%)" if cm[i, j] > 0 else "0"

sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', xticklabels=display_labels, yticklabels=display_labels,
            cbar_kws={'label': 'Normalized Recall'}, linewidths=0.5, linecolor='#e0e0e0')
plt.title('Confusion Matrix: Ancient Tamil Characters (SVM-Poly + Gabor/SIFT)', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Predicted Class', fontsize=11, fontweight='semibold')
plt.ylabel('True Class', fontsize=11, fontweight='semibold')
plt.tight_layout()
plt.show()

# Top Confused Pairs Table
df_confused = evaluate.identify_most_confused_pairs(cm, class_names, top_n=5)
print("Top Confused Character Pairs on Palm Leaves:")
display(df_confused)

## 7. Interactive Single-Character Inference (`predict.py`)
Pass any character image crop to produce predicted glyph, confidence score, and top-3 candidate probabilities.

In [ ]:
sample_test_path = list(config.DATA_DIR.glob("01_a/*.png"))[0]
best_class, best_conf, top_preds = predict.predict_single_character(
    image_path=str(sample_test_path),
    save_card=True
)

# Display the visual prediction card inline
pred_card_path = config.RESULTS_DIR / "prediction_output.png"
if pred_card_path.exists():
    from PIL import Image
    display(Image.open(pred_card_path))
else:
    print(f"Prediction: {best_class} ({config.TAMIL_GLYPH_MAP.get(best_class, '')}) | Confidence: {best_conf*100:.1f}%")

## 8. Authentic Historical Palm-Leaf Manuscript Processing (*Tolkāppiyam*)
Tests the end-to-end pipeline on an authentic 18th-century scanned digital manuscript of the classical **Tolkāppiyam (தொல்காப்பியம்)**:
1. Restores the real palm leaf strip via Sauvola binarization.
2. Segments character glyphs using **Contour-Based Convex Hull Bounding Boxes** (Paper 2).
3. Classifies each glyph crop using the trained classical model.
4. Generates an annotated recognition card.

In [ ]:
from PIL import Image

card_path, extracted_chars = process_manuscript.process_historical_manuscript(
    manuscript_path='manuscripts/tolkappiyam_palm_leaf.jpg',
    output_dir='results/manuscript_analysis',
    max_characters_to_extract=12
)

if card_path.exists():
    print("\nHistorical Tolkāppiyam Palm-Leaf Manuscript Analysis Card:")
    display(Image.open(card_path))
else:
    print('Manuscript card generated.')